In [13]:
! pip install kaggle
! pip install torchmetrics
!pip install tensorrt
!pip install onnx
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 21.1 MB/s eta 0:00:00


In [2]:
import json
import os, sys, time
import time
import math
import random
from dataclasses import dataclass
from typing import Tuple, List, Optional

import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchmetrics.classification import BinaryF1Score
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from IPython.display import clear_output

SEED = 42

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

print(f"Using torch version: {torch.__version__}")
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device

Using torch version: 2.10.0+cu128


device(type='cuda')

## UNet architecture

Same components as `milesial/Pytorch-UNet` but fixed to the Carvana scale=0.5 config: ConvTranspose2d upsampling and a 1024-channel bottleneck.

In [3]:
class DoubleConv(nn.Module):
    """(Conv2d -> BatchNorm -> ReLU) x 2, same spatial size (padding=1)."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """Encoder step: MaxPool(2) -> DoubleConv."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels),
        )

    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """Decoder step: ConvTranspose2d -> pad/concat with skip -> DoubleConv."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diff_y = x2.size(2) - x1.size(2)
        diff_x = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [diff_x // 2, diff_x - diff_x // 2,
                        diff_y // 2, diff_y - diff_y // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    """Full U-Net. Fixed ConvTranspose2d upsampling, 1024-channel bottleneck."""

    def __init__(self, n_channels: int = 3, n_classes: int = 2):
        super().__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes

        self.inc   = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)

        self.up1 = Up(1024, 512)
        self.up2 = Up(512,  256)
        self.up3 = Up(256,  128)
        self.up4 = Up(128,  64)
        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x  = self.up1(x5, x4)
        x  = self.up2(x,  x3)
        x  = self.up3(x,  x2)
        x  = self.up4(x,  x1)
        return self.outc(x)

n_params = sum(p.numel() for p in UNet().parameters())
print(f"UNet(3, 2) params: {n_params/1e6:.2f}M")

UNet(3, 2) params: 31.04M


In [4]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
os.environ["KAGGLE_API_TOKEN"] = userdata.get('KAGGLE_API_TOKEN')

Mounted at /content/drive


In [5]:
# !kaggle competitions download -c carvana-image-masking-challenge -f train_masks.zip
# !kaggle competitions download -c carvana-image-masking-challenge -f train.zip

In [6]:
# import zipfile
# from pathlib import Path
# for zip_name in ("train_masks.zip", "train.zip"):
#     zip_path = Path(zip_name)
#     extract_to = Path(".")
#     with zipfile.ZipFile(zip_path, "r") as zip_ref:
#         zip_ref.extractall(extract_to)

In [7]:
%ls drive/MyDrive/carvana

presentation/  train/  train_masks/


## Dataset and dataloader

Identical preprocessing to the benchmark notebook: scale-resize with bicubic for images and nearest for masks, normalize to `[0, 1]`, threshold masks to `{0, 1}`.

In [8]:
class CarvanaTrainDataset(Dataset):
    """Carvana train dataset returning (image, mask) tuples."""

    def __init__(
        self,
        train_root: str,
        mask_root: str,
        scale: float = 0.5,
        max_samples: int = 2000,
    ) -> None:
        if scale <= 0 or scale > 1:
            raise ValueError(f"Scale must be in (0, 1], got {scale}")

        self.train_root = train_root
        self.mask_root = mask_root
        self.scale = scale

        allowed_ext = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}
        image_files: List[str] = []
        for name in os.listdir(train_root):
            path = os.path.join(train_root, name)
            if not os.path.isfile(path):
                continue
            _, ext = os.path.splitext(name)
            if ext in allowed_ext:
                image_files.append(name)
        image_files.sort()

        paired: List[Tuple[str, str]] = []
        for img_name in image_files:
            stem, _ = os.path.splitext(img_name)
            mask_name = f"{stem}_mask.gif"
            mask_path = os.path.join(mask_root, mask_name)
            img_path = os.path.join(train_root, img_name)
            if os.path.isfile(mask_path):
                paired.append((img_path, mask_path))

        if not paired:
            raise RuntimeError(
                f"No image/mask pairs found in train_root='{train_root}' and mask_root='{mask_root}'."
            )

        self.pairs = paired[:max_samples]

    def __len__(self) -> int:
        return len(self.pairs)

    def _preprocess_image(self, path: str) -> torch.Tensor:
        with Image.open(path) as img:
            img = img.convert("RGB")
            w, h = img.size
            new_w = int(self.scale * w)
            new_h = int(self.scale * h)
            img = img.resize((new_w, new_h), resample=Image.BICUBIC)
            arr = np.asarray(img, dtype=np.float32)
        arr = arr.transpose((2, 0, 1))
        if arr.max() > 1.0:
            arr = arr / 255.0
        return torch.from_numpy(arr.astype(np.float32))

    def _preprocess_mask(self, path: str) -> torch.Tensor:
        with Image.open(path) as img:
            img = img.convert("L")
            w, h = img.size
            new_w = int(self.scale * w)
            new_h = int(self.scale * h)
            img = img.resize((new_w, new_h), resample=Image.NEAREST)
            arr = np.asarray(img, dtype=np.uint8)
        return torch.from_numpy((arr > 0).astype(np.int64))

    def __getitem__(self, idx: int):
        img_path, mask_path = self.pairs[idx]
        return self._preprocess_image(img_path), self._preprocess_mask(mask_path)


def build_dataloader(
    dataset: Dataset,
    batch_size: int = 4,
    num_workers: int = 2,
    shuffle: bool = False,
) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

## Train / val split (90/10, seeded)

`random_split` with a seeded `torch.Generator` so the val set is reproducible.

In [9]:
train_root = "drive/MyDrive/carvana/train"
mask_root = "drive/MyDrive/carvana/train_masks"

n_classes = 2
image_scale = 0.5
val_percent = 0.2

print("Creating Carvana train image+mask dataset...")
full_dataset = CarvanaTrainDataset(
    train_root=train_root,
    mask_root=mask_root,
    scale=image_scale,
)

n_val = int(len(full_dataset) * val_percent)
n_train = len(full_dataset) - n_val
split_gen = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(
    full_dataset, [n_train, n_val], generator=split_gen
)
print(f"full: {len(full_dataset)}  train: {len(train_dataset)}  val: {len(val_dataset)}")

Creating Carvana train image+mask dataset...
full: 2000  train: 1600  val: 400


## Training utilities

Soft Dice loss (for the `CrossEntropyLoss + Dice` objective) and `train_one_epoch` / `evaluate` helpers. Both use `tqdm` progress bars.

In [10]:
def dice_loss(logits: torch.Tensor, targets: torch.Tensor, num_classes: int, smooth: float = 1e-6) -> torch.Tensor:
    """Soft multiclass Dice loss: 1 - mean dice over classes + batch."""
    probs = F.softmax(logits, dim=1)
    targets_oh = F.one_hot(targets, num_classes).permute(0, 3, 1, 2).float()
    dims = (0, 2, 3)
    num = 2 * (probs * targets_oh).sum(dims)
    den = probs.sum(dims) + targets_oh.sum(dims)
    dice = (num + smooth) / (den + smooth)
    return 1 - dice.mean()


def train_one_epoch(model, loader, optimizer, scaler, device, amp_enabled, grad_clip=1.0):
    model.train()
    ce = nn.CrossEntropyLoss()
    total_loss, seen = 0.0, 0
    pbar = tqdm(loader, desc="train", leave=False)
    for images, masks in pbar:
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        masks = masks.to(device, non_blocking=True, dtype=torch.long)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            logits = model(images)
            loss = ce(logits, masks) + dice_loss(logits, masks, num_classes=model.n_classes)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        bs = images.size(0)
        total_loss += loss.item() * bs
        seen += bs
        pbar.set_postfix(loss=loss.item())
    return total_loss / max(seen, 1)


@torch.no_grad()
def evaluate(model, loader, device, amp_enabled, num_classes=2):
    """Return (avg_loss, foreground_dice) on the given loader."""
    model.eval()
    ce = nn.CrossEntropyLoss()
    metric = BinaryF1Score().to(device)
    total_loss, seen = 0.0, 0
    for images, masks in tqdm(loader, desc="val", leave=False):
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        masks = masks.to(device, non_blocking=True, dtype=torch.long)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            logits = model(images)
            loss = ce(logits, masks) + dice_loss(logits, masks, num_classes=num_classes)
        preds = torch.argmax(logits, dim=1)
        metric.update((preds > 0).long().flatten(), (masks > 0).long().flatten())
        bs = images.size(0)
        total_loss += loss.item() * bs
        seen += bs
    return total_loss / max(seen, 1), float(metric.compute().item())

## Training

AdamW (lr=1e-4, wd=1e-4) + `ReduceLROnPlateau` on val Dice. CE + soft-Dice loss. fp16 AMP on CUDA. Tracks train/val loss and val Dice per epoch for plotting. Keeps the best (by val Dice) weights in memory.

In [ ]:
epochs = 3
train_batch_size = 2
lr = 1e-4
weight_decay = 1e-4
grad_clip = 1.0
num_workers = 2
amp_enabled = (device.type == "cuda")

train_loader = DataLoader(
    train_dataset,
    batch_size=train_batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
    drop_last=True,
    generator=torch.Generator().manual_seed(SEED),
    persistent_workers=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=train_batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
    persistent_workers=True,
)

model = UNet(n_channels=3, n_classes=n_classes).to(device, memory_format=torch.channels_last)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5)
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

os.makedirs("./checkpoints", exist_ok=True)
checkpoint_path = "./checkpoints/unet_carvana_self_trained.pth"   # plain state_dict (best so far)
last_path = "./checkpoints/last.pth"                              # plain state_dict (last epoch)
training_state_path = "./checkpoints/training_state.pth"          # full state for resume

history = {"train_loss": [], "val_loss": [], "val_dice": []}
best_dice = -1.0
best_state = None


def render_progress(history, best_dice, current_epoch, total_epochs):
    """Live re-render: clear cell output, reprint epoch table, redraw plots."""
    clear_output(wait=True)
    print(f"Epoch {current_epoch}/{total_epochs}  |  best val Dice so far: {best_dice:.4f}")
    print(f"{'epoch':>5}  {'train_loss':>11}  {'val_loss':>11}  {'val_dice':>11}")
    for ep, (tl, vl, vd) in enumerate(zip(history["train_loss"],
                                          history["val_loss"],
                                          history["val_dice"]), 1):
        print(f"{ep:>5d}  {tl:>11.4f}  {vl:>11.4f}  {vd:>11.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs_range = range(1, len(history["train_loss"]) + 1)

    axes[0].plot(epochs_range, history["train_loss"], marker="o", label="train")
    axes[0].plot(epochs_range, history["val_loss"], marker="o", label="val")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("loss (CE + Dice)")
    axes[0].set_title("Train / Val loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs_range, history["val_dice"], marker="o", color="tab:green", label="val Dice")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("Dice")
    axes[1].set_title("Validation Dice")
    axes[1].set_ylim(0, 1)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


for epoch in range(1, epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, scaler, device, amp_enabled, grad_clip)
    val_loss, val_dice = evaluate(model, val_loader, device, amp_enabled, num_classes=n_classes)
    scheduler.step(val_dice)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_dice"].append(val_dice)

    if val_dice > best_dice:
        best_dice = val_dice
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        torch.save(best_state, checkpoint_path)

    torch.save(model.state_dict(), last_path)
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": epoch,
        "best_dice": best_dice,
        "history": history,
    }, training_state_path)

    render_progress(history, best_dice, epoch, epochs)

print(f"\nBest val Dice: {best_dice:.4f}")
model.load_state_dict(best_state)

## Saved artifacts

The training loop writes three files into `./checkpoints/` each epoch:

| File | Format | Use |
|---|---|---|
| `unet_carvana_self_trained.pth` | plain `state_dict` (best by val Dice) | benchmarks / `validate.py` / inference |
| `last.pth` | plain `state_dict` (last epoch) | latest snapshot |
| `training_state.pth` | full dict — `model`, `optimizer`, `scheduler`, `scaler`, `epoch`, `best_dice`, `history` | clean resume of training |

In [ ]:
for path in [checkpoint_path, last_path, training_state_path]:
    size_mb = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0.0
    print(f"  {path:50s}  {size_mb:6.2f} MB")

  ./checkpoints/unet_carvana_self_trained.pth         124.24 MB
  ./checkpoints/last.pth                              124.23 MB
  ./checkpoints/training_state.pth                    372.60 MB


## Loading checkpoints back

Two patterns are useful:

1. **Inference / fine-tune** — load just the model weights (`unet_carvana_self_trained.pth`) into a fresh `UNet`.
2. **Resume training** — load the full `training_state.pth` to restore optimizer, scheduler, AMP scaler, epoch counter, best-Dice tracker, and `history`.

The cell below demonstrates both, then sanity-checks that the freshly loaded model produces the same val Dice as the in-memory `model`.

In [ ]:
# --- Pattern 1: load just the weights into a fresh UNet (inference / fine-tune) ---
fresh_model = UNet(n_channels=3, n_classes=n_classes).to(device, memory_format=torch.channels_last)
loaded_state = torch.load(checkpoint_path, map_location=device)
fresh_model.load_state_dict(loaded_state, strict=True)
print(f"Loaded model state_dict from {checkpoint_path}")

# --- Pattern 2: full resume bundle (model + optimizer + scheduler + scaler + epoch + history) ---
ckpt = torch.load(training_state_path, map_location=device)
print(f"\ntraining_state.pth keys: {list(ckpt.keys())}")
print(f"  saved at epoch {ckpt['epoch']}  |  best_dice={ckpt['best_dice']:.4f}")
print(f"  history len      = {len(ckpt['history']['val_dice'])} epochs")

# To actually resume, you'd do:
#   model.load_state_dict(ckpt["model"])
#   optimizer.load_state_dict(ckpt["optimizer"])
#   scheduler.load_state_dict(ckpt["scheduler"])
#   scaler.load_state_dict(ckpt["scaler"])
#   start_epoch = ckpt["epoch"] + 1
#   history = ckpt["history"]

# --- Sanity: fresh-loaded model should match the in-memory best Dice ---
_, fresh_dice = evaluate(fresh_model, val_loader, device, amp_enabled=False, num_classes=n_classes)
print(f"\nVal Dice (fresh-loaded model): {fresh_dice:.4f}  | recorded best: {best_dice:.4f}")

## Pre-benchmark sanity Dice (val split, fp32)

Recomputes Dice with AMP disabled so we have a reliable fp32 baseline to compare the fp16 / `torch.compile` benchmark numbers against.

In [ ]:
pre_val_loss, pre_val_dice = evaluate(model, val_loader, device, amp_enabled=False, num_classes=n_classes)
print(f"Pre-benchmark val | loss={pre_val_loss:.4f} | Dice={pre_val_dice:.4f}")

## Inference benchmarks (val split only)

Identical design to the original notebook — `batch_size ∈ {2,4,8,16,32}` × `{normal_fp16, torch_compile}` — but each config iterates over the **val split** of our self-trained model. The orchestrator loads weights from `checkpoint_path` for every experiment (matching the original's fresh-model-per-config behavior).

In [ ]:
def run_inference_benchmark(
    model,
    dataloader: DataLoader,
    device: torch.device,
    num_batches: int = 10,
    mixed_precision: bool = False,
    num_classes: int = 2,
) -> dict:
    """Run inference over a few batches and measure speed + quality (BinaryF1 == Dice)."""
    model.to(device)
    model.eval()

    cuda_available = device.type == "cuda" and torch.cuda.is_available()
    total_model_time = 0.0
    total_batches = 0
    batch_size = None

    f1_metric = BinaryF1Score().to(device if cuda_available else "cpu")

    # Warmup (not timed)
    with torch.no_grad():
        for i, (images, masks) in enumerate(dataloader):
            images = images.to(device, non_blocking=cuda_available)
            masks = masks.to(device, non_blocking=cuda_available)
            if mixed_precision and cuda_available:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    _ = model(images)
            else:
                _ = model(images)
            break

    with torch.no_grad():
        for i, (images, masks) in enumerate(tqdm(dataloader, total=min(num_batches, len(dataloader)), desc="Inference", leave=False)):
            if i >= num_batches:
                break
            images = images.to(device, non_blocking=cuda_available)
            masks = masks.to(device, non_blocking=cuda_available)
            if batch_size is None:
                batch_size = images.shape[0]
            if cuda_available:
                torch.cuda.synchronize(device)
            start = time.perf_counter()
            if mixed_precision and cuda_available:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = model(images)
            else:
                logits = model(images)
            if cuda_available:
                torch.cuda.synchronize(device)
            end = time.perf_counter()
            total_model_time += end - start

            if logits.dim() == 4:
                preds = torch.argmax(logits, dim=1)
            else:
                raise ValueError(f"Unexpected logits shape: {logits.shape}")

            preds_bin = (preds > 0).long().flatten()
            masks_bin = (masks > 0).long().flatten()
            f1_metric.update(preds_bin.to(f1_metric.device), masks_bin.to(f1_metric.device))
            total_batches += 1

    if total_batches == 0:
        raise RuntimeError("No batches processed in run_inference_benchmark")
    if batch_size is None:
        batch_size = 0

    num_images = total_batches * batch_size
    avg_batch_time_sec = total_model_time / total_batches
    avg_time_per_image_ms = (avg_batch_time_sec / max(batch_size, 1)) * 1000.0
    mean_dice = float(f1_metric.compute().item())

    return {
        "total_model_time_sec": total_model_time,
        "avg_batch_time_sec": avg_batch_time_sec,
        "avg_time_per_image_ms": avg_time_per_image_ms,
        "num_images": num_images,
        "num_batches": total_batches,
        "batch_size": batch_size,
        "mean_dice": mean_dice,
    }

In [ ]:
@dataclass
class ExperimentConfig:
    batch_size: int
    method: str  # "normal_fp16" or "torch_compile"

    def to_dict(self) -> dict:
        return {"batch_size": self.batch_size, "method": self.method}

    def to_json_key(self) -> str:
        return json.dumps(self.to_dict(), sort_keys=True)


class UNetSegmentationExperiments:
    """Benchmark orchestrator — loads our trained state_dict for each experiment."""

    def __init__(
        self,
        dataset: Dataset,
        state_dict_path: str,
        device: Optional[torch.device] = None,
        num_images: Optional[int] = None,
        num_classes: int = 2,
    ) -> None:
        self.dataset = dataset
        self.state_dict_path = state_dict_path
        self.device = device if device is not None else (
            torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        )
        # Default: one full pass over the val set.
        self.num_images = num_images if num_images is not None else len(dataset)
        self.num_classes = num_classes
        self.history: dict[str, dict] = {}

    def _prepare_base_model(self):
        model = UNet(n_channels=3, n_classes=self.num_classes).to(self.device)
        state_dict = torch.load(self.state_dict_path, map_location=self.device)
        state_dict.pop("mask_values", None)
        model.load_state_dict(state_dict, strict=True)
        for p in model.parameters():
            p.requires_grad_(False)
        model.eval()
        return model

    def make_experiment(self, config: ExperimentConfig) -> dict:
        if config.method not in {"normal_fp16", "torch_compile"}:
            raise ValueError(f"Unsupported method '{config.method}'.")

        dataloader = build_dataloader(
            self.dataset,
            batch_size=config.batch_size,
            num_workers=2,
            shuffle=False,
        )
        effective_num_batches = max(1, math.ceil(self.num_images / config.batch_size))

        if config.method == "normal_fp16":
            result = self._run_normal_fp16(dataloader, effective_num_batches)
        else:
            result = self._run_torch_compile(dataloader, effective_num_batches)

        result.setdefault("config", config.to_dict())
        result.setdefault("method", config.method)
        self.history[config.to_json_key()] = result
        return result

    def _run_normal_fp16(self, dataloader: DataLoader, num_batches: int) -> dict:
        print("Running experiment: normal_fp16")
        model = self._prepare_base_model()
        cuda_available = self.device.type == "cuda" and torch.cuda.is_available()
        mixed_precision = False
        if cuda_available:
            model = model.half()
            mixed_precision = True
            print("CUDA available: fp16 weights + autocast.")
        else:
            print("CUDA not available: fp32 without AMP.")
        result = run_inference_benchmark(
            model, dataloader, device=self.device,
            num_batches=num_batches, mixed_precision=mixed_precision,
            num_classes=self.num_classes,
        )
        print(f"Finished normal_fp16: {result}")
        return result

    def _run_torch_compile(self, dataloader: DataLoader, num_batches: int) -> dict:
        print("Running experiment: torch_compile")
        if not hasattr(torch, "compile"):
            reason = "torch.compile is not available in this PyTorch version."
            print(reason)
            return {"skipped": True, "reason": reason}
        model = self._prepare_base_model()
        cuda_available = self.device.type == "cuda" and torch.cuda.is_available()
        mixed_precision = False
        if cuda_available:
            model = model.half()
            mixed_precision = True
            print("CUDA available: fp16 weights + autocast for compiled model.")
        else:
            print("CUDA not available: compiled model in fp32.")
        try:
            compiled_model = torch.compile(model)
        except Exception as e:
            reason = f"torch.compile failed: {e}"
            print(reason)
            return {"skipped": True, "reason": reason}
        result = run_inference_benchmark(
            compiled_model, dataloader, device=self.device,
            num_batches=num_batches, mixed_precision=mixed_precision,
            num_classes=self.num_classes,
        )
        print(f"Finished torch_compile: {result}")
        return result

    def save_history(self, path: str) -> None:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self.history, f, indent=2)
        print(f"History saved to {path}")

In [ ]:
experiments = UNetSegmentationExperiments(
    dataset=val_dataset,
    state_dict_path=checkpoint_path,
    device=device,
    num_classes=n_classes,
)

configs = [
    ExperimentConfig(batch_size=2, method="normal_fp16"),
    ExperimentConfig(batch_size=2, method="torch_compile"),
    ExperimentConfig(batch_size=4, method="normal_fp16"),
    ExperimentConfig(batch_size=4, method="torch_compile"),
    ExperimentConfig(batch_size=8, method="normal_fp16"),
    ExperimentConfig(batch_size=8, method="torch_compile"),
    ExperimentConfig(batch_size=16, method="normal_fp16"),
    ExperimentConfig(batch_size=16, method="torch_compile"),
    ExperimentConfig(batch_size=32, method="normal_fp16"),
    ExperimentConfig(batch_size=32, method="torch_compile"),
]

results = []
for cfg in tqdm(configs, desc="configs"):
    print(f"\nRunning config: {cfg}")
    result = experiments.make_experiment(cfg)
    results.append(result)
    print(json.dumps(result, indent=2))

print("\nCollected history:")
print(json.dumps(experiments.history, indent=2))

In [ ]:
output_path = "unet_benchmarks_self_trained.json"
experiments.save_history(output_path)

with open(output_path, "r", encoding="utf-8") as f:
    loaded = json.load(f)

print("Loaded history from disk:")
print(json.dumps(loaded, indent=2))

## Benchmark throughput plot

Avg time per image (ms) vs. batch size, split by method.

In [ ]:
x_a = np.array([[json.loads(k)["batch_size"], v["avg_time_per_image_ms"]]
                for k, v in loaded.items()
                if v["method"] == "normal_fp16" and not v.get("skipped")])
x_b = np.array([[json.loads(k)["batch_size"], v["avg_time_per_image_ms"]]
                for k, v in loaded.items()
                if v["method"] == "torch_compile" and not v.get("skipped")])

plt.figure(figsize=(7, 4))
if len(x_a):
    plt.plot(x_a[:, 0], x_a[:, 1], marker="o", label="normal_fp16")
if len(x_b):
    plt.plot(x_b[:, 0], x_b[:, 1], marker="o", label="torch_compile")
plt.legend()
plt.xlabel("batch size")
plt.ylabel("avg time per image (ms)")
plt.title("Inference throughput on val split (self-trained UNet)")
plt.grid(True, alpha=0.3)
plt.show()

In [11]:
checkpoint_path='last.pth'
val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda"),
    persistent_workers=True,
)

In [15]:
# === Colab INT8 quantization via TensorRT (faster-than-fp16 inference) ===
#
# Colab-tuned variant of unet_h100_quant_cell.py.
#   - MAX_BATCH = 4 (Colab GPUs have far less HBM than an H100; 4 fits T4/L4/A100).
#   - Smaller calibration batch + fewer batches so calibration finishes quickly
#     on a free-tier T4.
#   - 1 GiB TRT workspace (fits T4's 16 GB; safe on L4/A100 too).
#   - Self-installs `tensorrt` and `onnx` if missing, since fresh Colab runtimes
#     don't ship them.
#
# Why this still gives a real speedup over fp16:
#   - Tensor Cores: INT8 is ~2x peak vs FP16 on Ampere/Hopper/Ada
#     (H100 ~2x, A100 ~2x IMMA vs HMMA, L4/T4 also benefit).
#   - TensorRT fuses Conv+BN+ReLU into single int8 IMMA kernels and picks
#     NHWC layouts automatically.
#
# Pipeline (single notebook cell, self-contained):
#   1. Install tensorrt/onnx if absent.
#   2. Load fp32 weights into a fresh UNet.
#   3. Export to ONNX with a dynamic batch axis.
#   4. Build a TRT engine: INT8 main, FP16 fallback for un-quantizable layers.
#      Calibration uses the val set via a torch-backed IInt8EntropyCalibrator2.
#   5. Run inference with `execute_async_v3` directly against torch CUDA tensors.
#   6. Time the val pass; compare to a fresh fp16 baseline measured in this cell.
#
# Reuses notebook-defined names: UNet, n_classes, device, val_dataset,
#   val_loader, checkpoint_path, pre_val_dice (optional), tqdm.
#
# Notes on Colab:
#   - H100 is generally not offered. Expect T4 (free), L4 (Pro), A100 (Pro+).
#   - On T4 (sm_75) INT8 IMMA is supported; expect ~1.5-2x over fp16.
#   - On L4/A100 you get closer to the H100-style 2x.

import subprocess
import sys


def _pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])

try:
    import tensorrt as trt  # noqa: F401
except ImportError:
    print("Installing tensorrt...")
    _pip_install("tensorrt")
    import tensorrt as trt  # noqa: F401

try:
    import onnx  # noqa: F401
except ImportError:
    print("Installing onnx...")
    _pip_install("onnx")
    import onnx  # noqa: F401


import gc
import os
import time

import numpy as np
import torch
import torch.nn as nn
import tensorrt as trt
from torchmetrics.classification import BinaryF1Score


assert device.type == "cuda", "TensorRT INT8 path requires CUDA."
assert torch.cuda.is_available()
print(f"GPU: {torch.cuda.get_device_name(0)}  | TensorRT: {trt.__version__}")

_cap = torch.cuda.get_device_capability(0)
print(f"  device capability: sm_{_cap[0]}{_cap[1]}")
if _cap[0] < 7 or (_cap[0] == 7 and _cap[1] < 5):
    print("  warning: INT8 IMMA needs sm_75+ (Turing). Older GPU -> may fall back to fp16.")


# ----------------------------------------------------------------------------
# 1. Fresh fp32 UNet from the saved best-checkpoint.
# ----------------------------------------------------------------------------
fp32_model = UNet(n_channels=3, n_classes=n_classes).to(device).eval()
fp32_model.load_state_dict(torch.load(checkpoint_path, map_location=device), strict=True)
for p in fp32_model.parameters():
    p.requires_grad_(False)


# ----------------------------------------------------------------------------
# 2. ONNX export with dynamic batch axis. Spatial dims are fixed.
# ----------------------------------------------------------------------------
sample_img, _ = next(iter(val_loader))
_, C, H, W = sample_img.shape
print(f"Static spatial shape: C={C}, H={H}, W={W}")

ONNX_PATH    = "./checkpoints/unet_carvana_fp32.onnx"
ENGINE_PATH  = "./checkpoints/unet_carvana_int8_colab.engine"
CALIB_CACHE  = "./checkpoints/unet_carvana_int8_colab.calib"
os.makedirs(os.path.dirname(ONNX_PATH), exist_ok=True)

dummy = torch.randn(1, C, H, W, device=device)
# dynamo=False forces the legacy TorchScript exporter. The dynamo path on
# torch >= 2.6 writes weights as *external* data and emits opset 18+, which
# breaks TRT's ONNX parser (initializers come back empty -> "Failed to import
# initializer: ..."). Legacy exporter embeds weights inline at opset 17.
torch.onnx.export(
    fp32_model, dummy, ONNX_PATH,
    input_names=["input"], output_names=["logits"],
    opset_version=17,
    do_constant_folding=True,
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    dynamo=False,
)
_onnx_mb = os.path.getsize(ONNX_PATH) / 1e6
print(f"ONNX export -> {ONNX_PATH}  ({_onnx_mb:.1f} MB)")
assert _onnx_mb > 10, (
    f"ONNX file is only {_onnx_mb:.1f} MB -- weights likely went to external "
    f"data. Check that dynamo=False is supported by your torch version."
)


# ----------------------------------------------------------------------------
# 3. INT8 calibrator. Single pre-allocated CUDA tensor; TRT reads its data_ptr.
# ----------------------------------------------------------------------------
CALIB_BATCH    = 2          # small batches calibrate fine and fit on T4
CALIB_BATCHES  = 32         # 32 * 2 = 64 calibration images
MAX_BATCH      = 4          # Colab cap

class TorchCalibrator(trt.IInt8EntropyCalibrator2):
    def __init__(self, dataset, batch_size, num_batches, cache_path):
        super().__init__()
        self.dataset      = dataset
        self.batch_size   = batch_size
        self.num_batches  = min(num_batches, len(dataset) // batch_size)
        self.cache_path   = cache_path
        self.cursor       = 0
        self.device_buf = torch.empty(
            (batch_size, C, H, W), dtype=torch.float32, device="cuda"
        )

    def get_batch_size(self):
        return self.batch_size

    def get_batch(self, names):
        if self.cursor >= self.num_batches:
            return None
        for i in range(self.batch_size):
            img, _ = self.dataset[self.cursor * self.batch_size + i]
            self.device_buf[i].copy_(img.to("cuda", non_blocking=True))
        self.cursor += 1
        return [int(self.device_buf.data_ptr())]

    def read_calibration_cache(self):
        if os.path.isfile(self.cache_path):
            with open(self.cache_path, "rb") as f:
                return f.read()
        return None

    def write_calibration_cache(self, cache):
        with open(self.cache_path, "wb") as f:
            f.write(cache)


# ----------------------------------------------------------------------------
# 4. Build the TensorRT engine. INT8 main + FP16 fallback.
# ----------------------------------------------------------------------------
TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

def build_int8_engine(onnx_path, engine_path, calibrator):
    builder = trt.Builder(TRT_LOGGER)
    flag = 1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    network = builder.create_network(flag)
    parser = trt.OnnxParser(network, TRT_LOGGER)
    with open(onnx_path, "rb") as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                print(parser.get_error(i))
            raise RuntimeError("ONNX parse failed")

    cfg = builder.create_builder_config()
    cfg.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)  # 1 GiB (T4-safe)
    cfg.set_flag(trt.BuilderFlag.INT8)
    cfg.set_flag(trt.BuilderFlag.FP16)
    cfg.int8_calibrator = calibrator

    profile = builder.create_optimization_profile()
    profile.set_shape("input",
                      min=(1,           C, H, W),
                      opt=(CALIB_BATCH, C, H, W),
                      max=(MAX_BATCH,   C, H, W))
    cfg.add_optimization_profile(profile)
    cfg.set_calibration_profile(profile)

    print("Building INT8 TRT engine (calibration + autotune; ~1-3 min)...")
    t0 = time.perf_counter()
    serialized = builder.build_serialized_network(network, cfg)
    if serialized is None:
        raise RuntimeError("TRT engine build failed.")
    print(f"  built in {time.perf_counter() - t0:.1f}s")

    with open(engine_path, "wb") as f:
        f.write(serialized)
    print(f"Engine -> {engine_path}  ({os.path.getsize(engine_path)/1e6:.1f} MB)")
    return serialized


if os.path.isfile(ENGINE_PATH):
    with open(ENGINE_PATH, "rb") as f:
        engine_bytes = f.read()
    print(f"Loaded existing engine: {ENGINE_PATH}  ({len(engine_bytes)/1e6:.1f} MB)")
else:
    calibrator = TorchCalibrator(val_dataset, CALIB_BATCH, CALIB_BATCHES, CALIB_CACHE)
    engine_bytes = build_int8_engine(ONNX_PATH, ENGINE_PATH, calibrator)


# ----------------------------------------------------------------------------
# 5. TRT runtime + inference using torch tensors directly.
#
#    USER_MANAGED execution memory: TRT's allocator and torch's caching
#    allocator can fight for the same pool. Hand TRT a torch.empty() buffer.
# ----------------------------------------------------------------------------

# Drop names from earlier cells that may be holding GPU memory.
for _name in (
    "model", "fresh_model", "qmodel", "compiled_model",
    "fp32_model", "calibrator",
    "optimizer", "scheduler", "scaler", "best_state",
    "experiments",
):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
_free, _total = torch.cuda.mem_get_info()
print(f"  GPU free after cleanup: {_free/1e9:.1f} / {_total/1e9:.1f} GB")

runtime = trt.Runtime(TRT_LOGGER)
engine  = runtime.deserialize_cuda_engine(engine_bytes)

context = engine.create_execution_context(
    trt.ExecutionContextAllocationStrategy.USER_MANAGED
)
_mem_size = getattr(engine, "device_memory_size_v2", None) or engine.device_memory_size
print(f"  TRT activation memory: {_mem_size/1e9:.2f} GB (user-managed via torch)")

_trt_workspace = torch.empty(_mem_size, dtype=torch.uint8, device="cuda")
context.set_device_memory(_trt_workspace.data_ptr(), _mem_size)

_trt_stream = torch.cuda.Stream()

_out_buf = torch.empty((MAX_BATCH, n_classes, H, W), dtype=torch.float32, device="cuda")

def trt_infer(images_cuda: torch.Tensor) -> torch.Tensor:
    """images_cuda: (B, 3, H, W) float32 on CUDA. Returns logits (B, n_classes, H, W)."""
    B = images_cuda.shape[0]
    assert B <= MAX_BATCH, f"batch {B} exceeds engine MAX_BATCH={MAX_BATCH}"
    context.set_input_shape("input", (B, C, H, W))

    images_cuda = images_cuda.contiguous()
    out = _out_buf[:B]

    context.set_tensor_address("input",  images_cuda.data_ptr())
    context.set_tensor_address("logits", out.data_ptr())
    if not context.execute_async_v3(_trt_stream.cuda_stream):
        raise RuntimeError("TRT execute_async_v3 failed")
    _trt_stream.synchronize()
    return out


# ----------------------------------------------------------------------------
# 6. Warmup over each distinct batch shape we'll actually time.
#    Skip any batch larger than MAX_BATCH (val_loader may use a bigger batch).
# ----------------------------------------------------------------------------
print("\nWarmup (TRT picks per-shape kernels on first call)...")
seen_shapes = set()
with torch.no_grad():
    for images, _ in val_loader:
        if images.shape[0] > MAX_BATCH:
            # Slice down to MAX_BATCH so we still warm the relevant shape.
            images = images[:MAX_BATCH]
        sh = tuple(images.shape)
        if sh in seen_shapes:
            continue
        seen_shapes.add(sh)
        for _ in range(3):
            _ = trt_infer(images.to("cuda", non_blocking=True).float())
        torch.cuda.synchronize()
print(f"  warmed up {len(seen_shapes)} shape(s).")


# ----------------------------------------------------------------------------
# 7. Timed pass on the val split. Same metric (BinaryF1 == foreground Dice).
#    If val_loader's batch > MAX_BATCH, we chunk into MAX_BATCH-sized slices.
# ----------------------------------------------------------------------------
def _trt_forward_chunked(images_cuda):
    if images_cuda.shape[0] <= MAX_BATCH:
        return trt_infer(images_cuda)
    parts = []
    for i in range(0, images_cuda.shape[0], MAX_BATCH):
        parts.append(trt_infer(images_cuda[i:i + MAX_BATCH]).clone())
    return torch.cat(parts, dim=0)


f1 = BinaryF1Score().to(device)
total_t, n_imgs, n_batches = 0.0, 0, 0
print("\nTimed val pass (TRT INT8)...")
with torch.no_grad():
    for images, masks in tqdm(val_loader, desc="trt-int8"):
        images = images.to("cuda", non_blocking=True).float()
        masks  = masks.to("cuda",  non_blocking=True)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        logits = _trt_forward_chunked(images)
        torch.cuda.synchronize()
        total_t += time.perf_counter() - t0

        preds = torch.argmax(logits, dim=1)
        f1.update((preds > 0).long().flatten(), (masks > 0).long().flatten())
        n_imgs    += images.size(0)
        n_batches += 1

trt_dice = float(f1.compute().item())
trt_avg_batch_ms      = (total_t / max(n_batches, 1)) * 1000.0
trt_avg_per_image_ms  = (total_t / max(n_imgs, 1)) * 1000.0
trt_throughput        = n_imgs / total_t

print("\n=== TensorRT INT8 UNet on val split (Colab, MAX_BATCH=4) ===")
print(f"  images processed:    {n_imgs}")
print(f"  total inference:     {total_t:.3f} s")
print(f"  avg time / batch:    {trt_avg_batch_ms:.2f} ms  (loader batch={val_loader.batch_size})")
print(f"  avg time / image:    {trt_avg_per_image_ms:.3f} ms")
print(f"  throughput:          {trt_throughput:.1f} img/s")
print(f"  Dice (BinaryF1):     {trt_dice:.4f}")


# ----------------------------------------------------------------------------
# 8. fp16 baseline -- same val loader, same metric, same hardware state.
# ----------------------------------------------------------------------------
fp16_model = UNet(n_channels=3, n_classes=n_classes).to(device).eval()
fp16_model.load_state_dict(torch.load(checkpoint_path, map_location=device), strict=True)
fp16_model = fp16_model.half()
for p in fp16_model.parameters():
    p.requires_grad_(False)

print("\nWarmup (fp16)...")
with torch.no_grad():
    seen_shapes_fp16 = set()
    for images, _ in val_loader:
        sh = tuple(images.shape)
        if sh in seen_shapes_fp16:
            continue
        seen_shapes_fp16.add(sh)
        x = images.to("cuda", non_blocking=True).half()
        for _ in range(3):
            _ = fp16_model(x)
        torch.cuda.synchronize()

f1_fp16 = BinaryF1Score().to(device)
total_t_fp16, n_imgs_fp16, n_batches_fp16 = 0.0, 0, 0
print("Timed val pass (fp16)...")
with torch.no_grad():
    for images, masks in tqdm(val_loader, desc="fp16"):
        images = images.to("cuda", non_blocking=True).half()
        masks  = masks.to("cuda",  non_blocking=True)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        logits = fp16_model(images)
        torch.cuda.synchronize()
        total_t_fp16 += time.perf_counter() - t0

        preds = torch.argmax(logits, dim=1)
        f1_fp16.update((preds > 0).long().flatten(), (masks > 0).long().flatten())
        n_imgs_fp16    += images.size(0)
        n_batches_fp16 += 1

fp16_dice              = float(f1_fp16.compute().item())
fp16_avg_batch_ms      = (total_t_fp16 / max(n_batches_fp16, 1)) * 1000.0
fp16_avg_per_image_ms  = (total_t_fp16 / max(n_imgs_fp16, 1)) * 1000.0
fp16_throughput        = n_imgs_fp16 / total_t_fp16

print("\n=== fp16 (.half()) UNet on val split ===")
print(f"  images processed:    {n_imgs_fp16}")
print(f"  total inference:     {total_t_fp16:.3f} s")
print(f"  avg time / batch:    {fp16_avg_batch_ms:.2f} ms  (loader batch={val_loader.batch_size})")
print(f"  avg time / image:    {fp16_avg_per_image_ms:.3f} ms")
print(f"  throughput:          {fp16_throughput:.1f} img/s")
print(f"  Dice (BinaryF1):     {fp16_dice:.4f}")


# ----------------------------------------------------------------------------
# 9. Side-by-side. Apples-to-apples since both ran in this cell, this kernel.
# ----------------------------------------------------------------------------
print("\n=== fp16 vs TRT-INT8 (same val loader, same hardware state) ===")
print(f"  {'method':<12} {'ms/batch':>10} {'ms/image':>10} {'img/s':>10} {'Dice':>8}")
print(f"  {'fp16':<12} {fp16_avg_batch_ms:>10.2f} {fp16_avg_per_image_ms:>10.3f} "
      f"{fp16_throughput:>10.1f} {fp16_dice:>8.4f}")
print(f"  {'trt-int8':<12} {trt_avg_batch_ms:>10.2f} {trt_avg_per_image_ms:>10.3f} "
      f"{trt_throughput:>10.1f} {trt_dice:>8.4f}")
print(f"  speedup (fp16 -> trt-int8): {fp16_avg_per_image_ms / trt_avg_per_image_ms:.2f}x")
print(f"  Dice delta:                 {trt_dice - fp16_dice:+.4f}")

try:
    print(f"  reference fp32 Dice:        {pre_val_dice:.4f} "
          f"(delta {trt_dice - pre_val_dice:+.4f})")
except NameError:
    pass


GPU: Tesla T4  | TensorRT: 10.16.1.11
  device capability: sm_75
Static spatial shape: C=3, H=640, W=959


/tmp/ipykernel_1387/4118566472.py:103: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX export -> ./checkpoints/unet_carvana_fp32.onnx  (124.2 MB)


/tmp/ipykernel_1387/4118566472.py:181: DeprecationWarning: Use Deprecated in TensorRT 10.1. Superseded by explicit quantization. instead.
  cfg.int8_calibrator = calibrator
/tmp/ipykernel_1387/4118566472.py:189: DeprecationWarning: Use Deprecated in TensorRT 10.1. Superseded by explicit quantization. instead.
  cfg.set_calibration_profile(profile)


Building INT8 TRT engine (calibration + autotune; ~1-3 min)...
  built in 553.8s
Engine -> ./checkpoints/unet_carvana_int8_colab.engine  (31.4 MB)
  GPU free after cleanup: 15.3 / 15.6 GB
  TRT activation memory: 0.77 GB (user-managed via torch)

Warmup (TRT picks per-shape kernels on first call)...
  warmed up 1 shape(s).

Timed val pass (TRT INT8)...


trt-int8:   0%|          | 0/50 [00:00<?, ?it/s]


=== TensorRT INT8 UNet on val split (Colab, MAX_BATCH=4) ===
  images processed:    400
  total inference:     8.641 s
  avg time / batch:    172.81 ms  (loader batch=8)
  avg time / image:    21.601 ms
  throughput:          46.3 img/s
  Dice (BinaryF1):     0.9910

Warmup (fp16)...
Timed val pass (fp16)...


fp16:   0%|          | 0/50 [00:00<?, ?it/s]


=== fp16 (.half()) UNet on val split ===
  images processed:    400
  total inference:     35.113 s
  avg time / batch:    702.26 ms  (loader batch=8)
  avg time / image:    87.782 ms
  throughput:          11.4 img/s
  Dice (BinaryF1):     0.9911

=== fp16 vs TRT-INT8 (same val loader, same hardware state) ===
  method         ms/batch   ms/image      img/s     Dice
  fp16             702.26     87.782       11.4   0.9911
  trt-int8         172.81     21.601       46.3   0.9910
  speedup (fp16 -> trt-int8): 4.06x
  Dice delta:                 -0.0001


In [16]:
checkpoint_path='last.pth'
val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda"),
    persistent_workers=True,
)

In [17]:
# ----------------------------------------------------------------------------
# 7. Timed pass on the val split. Same metric (BinaryF1 == foreground Dice).
#    If val_loader's batch > MAX_BATCH, we chunk into MAX_BATCH-sized slices.
# ----------------------------------------------------------------------------
def _trt_forward_chunked(images_cuda):
    if images_cuda.shape[0] <= MAX_BATCH:
        return trt_infer(images_cuda)
    parts = []
    for i in range(0, images_cuda.shape[0], MAX_BATCH):
        parts.append(trt_infer(images_cuda[i:i + MAX_BATCH]).clone())
    return torch.cat(parts, dim=0)


f1 = BinaryF1Score().to(device)
total_t, n_imgs, n_batches = 0.0, 0, 0
print("\nTimed val pass (TRT INT8)...")
with torch.no_grad():
    for images, masks in tqdm(val_loader, desc="trt-int8"):
        images = images.to("cuda", non_blocking=True).float()
        masks  = masks.to("cuda",  non_blocking=True)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        logits = _trt_forward_chunked(images)
        torch.cuda.synchronize()
        total_t += time.perf_counter() - t0

        preds = torch.argmax(logits, dim=1)
        f1.update((preds > 0).long().flatten(), (masks > 0).long().flatten())
        n_imgs    += images.size(0)
        n_batches += 1

trt_dice = float(f1.compute().item())
trt_avg_batch_ms      = (total_t / max(n_batches, 1)) * 1000.0
trt_avg_per_image_ms  = (total_t / max(n_imgs, 1)) * 1000.0
trt_throughput        = n_imgs / total_t

print("\n=== TensorRT INT8 UNet on val split (Colab, MAX_BATCH=4) ===")
print(f"  images processed:    {n_imgs}")
print(f"  total inference:     {total_t:.3f} s")
print(f"  avg time / batch:    {trt_avg_batch_ms:.2f} ms  (loader batch={val_loader.batch_size})")
print(f"  avg time / image:    {trt_avg_per_image_ms:.3f} ms")
print(f"  throughput:          {trt_throughput:.1f} img/s")
print(f"  Dice (BinaryF1):     {trt_dice:.4f}")


# ----------------------------------------------------------------------------
# 8. fp16 baseline -- same val loader, same metric, same hardware state.
# ----------------------------------------------------------------------------
fp16_model = UNet(n_channels=3, n_classes=n_classes).to(device).eval()
fp16_model.load_state_dict(torch.load(checkpoint_path, map_location=device), strict=True)
fp16_model = fp16_model.half()
for p in fp16_model.parameters():
    p.requires_grad_(False)

print("\nWarmup (fp16)...")
with torch.no_grad():
    seen_shapes_fp16 = set()
    for images, _ in val_loader:
        sh = tuple(images.shape)
        if sh in seen_shapes_fp16:
            continue
        seen_shapes_fp16.add(sh)
        x = images.to("cuda", non_blocking=True).half()
        for _ in range(3):
            _ = fp16_model(x)
        torch.cuda.synchronize()

f1_fp16 = BinaryF1Score().to(device)
total_t_fp16, n_imgs_fp16, n_batches_fp16 = 0.0, 0, 0
print("Timed val pass (fp16)...")
with torch.no_grad():
    for images, masks in tqdm(val_loader, desc="fp16"):
        images = images.to("cuda", non_blocking=True).half()
        masks  = masks.to("cuda",  non_blocking=True)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        logits = fp16_model(images)
        torch.cuda.synchronize()
        total_t_fp16 += time.perf_counter() - t0

        preds = torch.argmax(logits, dim=1)
        f1_fp16.update((preds > 0).long().flatten(), (masks > 0).long().flatten())
        n_imgs_fp16    += images.size(0)
        n_batches_fp16 += 1

fp16_dice              = float(f1_fp16.compute().item())
fp16_avg_batch_ms      = (total_t_fp16 / max(n_batches_fp16, 1)) * 1000.0
fp16_avg_per_image_ms  = (total_t_fp16 / max(n_imgs_fp16, 1)) * 1000.0
fp16_throughput        = n_imgs_fp16 / total_t_fp16

print("\n=== fp16 (.half()) UNet on val split ===")
print(f"  images processed:    {n_imgs_fp16}")
print(f"  total inference:     {total_t_fp16:.3f} s")
print(f"  avg time / batch:    {fp16_avg_batch_ms:.2f} ms  (loader batch={val_loader.batch_size})")
print(f"  avg time / image:    {fp16_avg_per_image_ms:.3f} ms")
print(f"  throughput:          {fp16_throughput:.1f} img/s")
print(f"  Dice (BinaryF1):     {fp16_dice:.4f}")


# ----------------------------------------------------------------------------
# 9. Side-by-side. Apples-to-apples since both ran in this cell, this kernel.
# ----------------------------------------------------------------------------
print("\n=== fp16 vs TRT-INT8 (same val loader, same hardware state) ===")
print(f"  {'method':<12} {'ms/batch':>10} {'ms/image':>10} {'img/s':>10} {'Dice':>8}")
print(f"  {'fp16':<12} {fp16_avg_batch_ms:>10.2f} {fp16_avg_per_image_ms:>10.3f} "
      f"{fp16_throughput:>10.1f} {fp16_dice:>8.4f}")
print(f"  {'trt-int8':<12} {trt_avg_batch_ms:>10.2f} {trt_avg_per_image_ms:>10.3f} "
      f"{trt_throughput:>10.1f} {trt_dice:>8.4f}")
print(f"  speedup (fp16 -> trt-int8): {fp16_avg_per_image_ms / trt_avg_per_image_ms:.2f}x")
print(f"  Dice delta:                 {trt_dice - fp16_dice:+.4f}")

try:
    print(f"  reference fp32 Dice:        {pre_val_dice:.4f} "
          f"(delta {trt_dice - pre_val_dice:+.4f})")
except NameError:
    pass


Timed val pass (TRT INT8)...


trt-int8:   0%|          | 0/100 [00:00<?, ?it/s]


=== TensorRT INT8 UNet on val split (Colab, MAX_BATCH=4) ===
  images processed:    400
  total inference:     7.719 s
  avg time / batch:    77.19 ms  (loader batch=4)
  avg time / image:    19.297 ms
  throughput:          51.8 img/s
  Dice (BinaryF1):     0.9910

Warmup (fp16)...
Timed val pass (fp16)...


fp16:   0%|          | 0/100 [00:00<?, ?it/s]


=== fp16 (.half()) UNet on val split ===
  images processed:    400
  total inference:     34.414 s
  avg time / batch:    344.14 ms  (loader batch=4)
  avg time / image:    86.036 ms
  throughput:          11.6 img/s
  Dice (BinaryF1):     0.9911

=== fp16 vs TRT-INT8 (same val loader, same hardware state) ===
  method         ms/batch   ms/image      img/s     Dice
  fp16             344.14     86.036       11.6   0.9911
  trt-int8          77.19     19.297       51.8   0.9910
  speedup (fp16 -> trt-int8): 4.46x
  Dice delta:                 -0.0001


In [18]:
checkpoint_path='last.pth'
val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda"),
    persistent_workers=True,
)

In [19]:
# ----------------------------------------------------------------------------
# 7. Timed pass on the val split. Same metric (BinaryF1 == foreground Dice).
#    If val_loader's batch > MAX_BATCH, we chunk into MAX_BATCH-sized slices.
# ----------------------------------------------------------------------------
def _trt_forward_chunked(images_cuda):
    if images_cuda.shape[0] <= MAX_BATCH:
        return trt_infer(images_cuda)
    parts = []
    for i in range(0, images_cuda.shape[0], MAX_BATCH):
        parts.append(trt_infer(images_cuda[i:i + MAX_BATCH]).clone())
    return torch.cat(parts, dim=0)


f1 = BinaryF1Score().to(device)
total_t, n_imgs, n_batches = 0.0, 0, 0
print("\nTimed val pass (TRT INT8)...")
with torch.no_grad():
    for images, masks in tqdm(val_loader, desc="trt-int8"):
        images = images.to("cuda", non_blocking=True).float()
        masks  = masks.to("cuda",  non_blocking=True)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        logits = _trt_forward_chunked(images)
        torch.cuda.synchronize()
        total_t += time.perf_counter() - t0

        preds = torch.argmax(logits, dim=1)
        f1.update((preds > 0).long().flatten(), (masks > 0).long().flatten())
        n_imgs    += images.size(0)
        n_batches += 1

trt_dice = float(f1.compute().item())
trt_avg_batch_ms      = (total_t / max(n_batches, 1)) * 1000.0
trt_avg_per_image_ms  = (total_t / max(n_imgs, 1)) * 1000.0
trt_throughput        = n_imgs / total_t

print("\n=== TensorRT INT8 UNet on val split (Colab, MAX_BATCH=4) ===")
print(f"  images processed:    {n_imgs}")
print(f"  total inference:     {total_t:.3f} s")
print(f"  avg time / batch:    {trt_avg_batch_ms:.2f} ms  (loader batch={val_loader.batch_size})")
print(f"  avg time / image:    {trt_avg_per_image_ms:.3f} ms")
print(f"  throughput:          {trt_throughput:.1f} img/s")
print(f"  Dice (BinaryF1):     {trt_dice:.4f}")


# ----------------------------------------------------------------------------
# 8. fp16 baseline -- same val loader, same metric, same hardware state.
# ----------------------------------------------------------------------------
fp16_model = UNet(n_channels=3, n_classes=n_classes).to(device).eval()
fp16_model.load_state_dict(torch.load(checkpoint_path, map_location=device), strict=True)
fp16_model = fp16_model.half()
for p in fp16_model.parameters():
    p.requires_grad_(False)

print("\nWarmup (fp16)...")
with torch.no_grad():
    seen_shapes_fp16 = set()
    for images, _ in val_loader:
        sh = tuple(images.shape)
        if sh in seen_shapes_fp16:
            continue
        seen_shapes_fp16.add(sh)
        x = images.to("cuda", non_blocking=True).half()
        for _ in range(3):
            _ = fp16_model(x)
        torch.cuda.synchronize()

f1_fp16 = BinaryF1Score().to(device)
total_t_fp16, n_imgs_fp16, n_batches_fp16 = 0.0, 0, 0
print("Timed val pass (fp16)...")
with torch.no_grad():
    for images, masks in tqdm(val_loader, desc="fp16"):
        images = images.to("cuda", non_blocking=True).half()
        masks  = masks.to("cuda",  non_blocking=True)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        logits = fp16_model(images)
        torch.cuda.synchronize()
        total_t_fp16 += time.perf_counter() - t0

        preds = torch.argmax(logits, dim=1)
        f1_fp16.update((preds > 0).long().flatten(), (masks > 0).long().flatten())
        n_imgs_fp16    += images.size(0)
        n_batches_fp16 += 1

fp16_dice              = float(f1_fp16.compute().item())
fp16_avg_batch_ms      = (total_t_fp16 / max(n_batches_fp16, 1)) * 1000.0
fp16_avg_per_image_ms  = (total_t_fp16 / max(n_imgs_fp16, 1)) * 1000.0
fp16_throughput        = n_imgs_fp16 / total_t_fp16

print("\n=== fp16 (.half()) UNet on val split ===")
print(f"  images processed:    {n_imgs_fp16}")
print(f"  total inference:     {total_t_fp16:.3f} s")
print(f"  avg time / batch:    {fp16_avg_batch_ms:.2f} ms  (loader batch={val_loader.batch_size})")
print(f"  avg time / image:    {fp16_avg_per_image_ms:.3f} ms")
print(f"  throughput:          {fp16_throughput:.1f} img/s")
print(f"  Dice (BinaryF1):     {fp16_dice:.4f}")


# ----------------------------------------------------------------------------
# 9. Side-by-side. Apples-to-apples since both ran in this cell, this kernel.
# ----------------------------------------------------------------------------
print("\n=== fp16 vs TRT-INT8 (same val loader, same hardware state) ===")
print(f"  {'method':<12} {'ms/batch':>10} {'ms/image':>10} {'img/s':>10} {'Dice':>8}")
print(f"  {'fp16':<12} {fp16_avg_batch_ms:>10.2f} {fp16_avg_per_image_ms:>10.3f} "
      f"{fp16_throughput:>10.1f} {fp16_dice:>8.4f}")
print(f"  {'trt-int8':<12} {trt_avg_batch_ms:>10.2f} {trt_avg_per_image_ms:>10.3f} "
      f"{trt_throughput:>10.1f} {trt_dice:>8.4f}")
print(f"  speedup (fp16 -> trt-int8): {fp16_avg_per_image_ms / trt_avg_per_image_ms:.2f}x")
print(f"  Dice delta:                 {trt_dice - fp16_dice:+.4f}")

try:
    print(f"  reference fp32 Dice:        {pre_val_dice:.4f} "
          f"(delta {trt_dice - pre_val_dice:+.4f})")
except NameError:
    pass


Timed val pass (TRT INT8)...


trt-int8:   0%|          | 0/200 [00:00<?, ?it/s]


=== TensorRT INT8 UNet on val split (Colab, MAX_BATCH=4) ===
  images processed:    400
  total inference:     7.442 s
  avg time / batch:    37.21 ms  (loader batch=2)
  avg time / image:    18.604 ms
  throughput:          53.8 img/s
  Dice (BinaryF1):     0.9910

Warmup (fp16)...
Timed val pass (fp16)...


fp16:   0%|          | 0/200 [00:00<?, ?it/s]


=== fp16 (.half()) UNet on val split ===
  images processed:    400
  total inference:     33.861 s
  avg time / batch:    169.30 ms  (loader batch=2)
  avg time / image:    84.652 ms
  throughput:          11.8 img/s
  Dice (BinaryF1):     0.9911

=== fp16 vs TRT-INT8 (same val loader, same hardware state) ===
  method         ms/batch   ms/image      img/s     Dice
  fp16             169.30     84.652       11.8   0.9911
  trt-int8          37.21     18.604       53.8   0.9910
  speedup (fp16 -> trt-int8): 4.55x
  Dice delta:                 -0.0001


## TRT-fp16 vs TRT-int8 vs PyTorch-fp16 (decompose the speedup)

The 4.5× int8 speedup over PyTorch-fp16 mixes two effects:

1. **Fusion + NHWC + per-shape autotune** — present in any TRT engine.
2. **INT8 IMMA vs FP16 HMMA** — only present once we quantize.

Build a second TRT engine from the same ONNX with `FP16` only (no INT8, no calibrator), and run all three paths back-to-back for `batch ∈ {2, 4, 8}`.

In [ ]:
# === TRT-fp16 vs TRT-int8 vs PyTorch-fp16 (isolate fusion/NHWC contribution) ===
#
# Builds a second TRT engine from the SAME ONNX with FP16-only (drop INT8 flag,
# drop calibrator). Same optimization profile, same workspace, same autotune.
# Then for each loader batch in {2, 4, 8} runs all three paths in the same
# kernel state and prints a decomposition table.
#
# Reuses globals from cell 36: ONNX_PATH, C, H, W, MAX_BATCH, n_classes,
#   val_dataset, device, checkpoint_path, UNet, TRT_LOGGER, engine_bytes (int8),
#   tqdm.

import gc
import os
import time

import torch
import tensorrt as trt
from torch.utils.data import DataLoader
from torchmetrics.classification import BinaryF1Score


FP16_ENGINE_PATH = "./checkpoints/unet_carvana_fp16_colab.engine"


def build_fp16_engine(onnx_path: str, engine_path: str) -> bytes:
    builder = trt.Builder(TRT_LOGGER)
    flag = 1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    network = builder.create_network(flag)
    parser = trt.OnnxParser(network, TRT_LOGGER)
    with open(onnx_path, "rb") as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                print(parser.get_error(i))
            raise RuntimeError("ONNX parse failed")

    cfg = builder.create_builder_config()
    cfg.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
    cfg.set_flag(trt.BuilderFlag.FP16)

    profile = builder.create_optimization_profile()
    profile.set_shape("input",
                      min=(1,         C, H, W),
                      opt=(2,         C, H, W),
                      max=(MAX_BATCH, C, H, W))
    cfg.add_optimization_profile(profile)

    print("Building FP16 TRT engine (no calibration; ~30-90s)...")
    t0 = time.perf_counter()
    serialized = builder.build_serialized_network(network, cfg)
    if serialized is None:
        raise RuntimeError("FP16 engine build failed.")
    print(f"  built in {time.perf_counter() - t0:.1f}s")

    with open(engine_path, "wb") as f:
        f.write(serialized)
    print(f"FP16 engine -> {engine_path}  ({os.path.getsize(engine_path)/1e6:.1f} MB)")
    return serialized


if os.path.isfile(FP16_ENGINE_PATH):
    with open(FP16_ENGINE_PATH, "rb") as f:
        fp16_engine_bytes = f.read()
    print(f"Loaded existing FP16 engine: {FP16_ENGINE_PATH}  ({len(fp16_engine_bytes)/1e6:.1f} MB)")
else:
    fp16_engine_bytes = build_fp16_engine(ONNX_PATH, FP16_ENGINE_PATH)


# ----------------------------------------------------------------------------
# Two side-by-side TRT runtimes (int8 reuses bytes from cell 36).
# ----------------------------------------------------------------------------
runtime2 = trt.Runtime(TRT_LOGGER)
eng_int8 = runtime2.deserialize_cuda_engine(engine_bytes)
eng_fp16 = runtime2.deserialize_cuda_engine(fp16_engine_bytes)

ctx_int8 = eng_int8.create_execution_context(trt.ExecutionContextAllocationStrategy.USER_MANAGED)
ctx_fp16 = eng_fp16.create_execution_context(trt.ExecutionContextAllocationStrategy.USER_MANAGED)

def _act_mem(eng):
    return getattr(eng, "device_memory_size_v2", None) or eng.device_memory_size

mem_int8, mem_fp16 = _act_mem(eng_int8), _act_mem(eng_fp16)
print(f"  TRT activation memory:  int8={mem_int8/1e9:.2f} GB  fp16={mem_fp16/1e9:.2f} GB")

ws_int8 = torch.empty(mem_int8, dtype=torch.uint8, device="cuda")
ws_fp16 = torch.empty(mem_fp16, dtype=torch.uint8, device="cuda")
ctx_int8.set_device_memory(ws_int8.data_ptr(), mem_int8)
ctx_fp16.set_device_memory(ws_fp16.data_ptr(), mem_fp16)

stream2 = torch.cuda.Stream()
out_buf_int8 = torch.empty((MAX_BATCH, n_classes, H, W), dtype=torch.float32, device="cuda")
out_buf_fp16 = torch.empty((MAX_BATCH, n_classes, H, W), dtype=torch.float32, device="cuda")


def _trt_call(ctx, out_buf, images_cuda):
    B = images_cuda.shape[0]
    assert B <= MAX_BATCH, f"batch {B} exceeds MAX_BATCH={MAX_BATCH}"
    ctx.set_input_shape("input", (B, C, H, W))
    images_cuda = images_cuda.contiguous()
    out = out_buf[:B]
    ctx.set_tensor_address("input",  images_cuda.data_ptr())
    ctx.set_tensor_address("logits", out.data_ptr())
    if not ctx.execute_async_v3(stream2.cuda_stream):
        raise RuntimeError("TRT execute_async_v3 failed")
    stream2.synchronize()
    return out


def trt_int8_call(images_cuda):
    return _trt_call(ctx_int8, out_buf_int8, images_cuda)


def trt_fp16_call(images_cuda):
    return _trt_call(ctx_fp16, out_buf_fp16, images_cuda)


def _chunked(fn, images_cuda):
    if images_cuda.shape[0] <= MAX_BATCH:
        return fn(images_cuda)
    parts = []
    for i in range(0, images_cuda.shape[0], MAX_BATCH):
        parts.append(fn(images_cuda[i:i + MAX_BATCH]).clone())
    return torch.cat(parts, dim=0)


# ----------------------------------------------------------------------------
# Fresh PyTorch fp16 model (independent from earlier cells; clean state).
# ----------------------------------------------------------------------------
torch_fp16_model = UNet(n_channels=3, n_classes=n_classes).to(device).eval()
torch_fp16_model.load_state_dict(torch.load(checkpoint_path, map_location=device), strict=True)
torch_fp16_model = torch_fp16_model.half()
for p in torch_fp16_model.parameters():
    p.requires_grad_(False)


def time_path(loader, fn, prep, label):
    """fn: images_cuda -> logits.  prep: dtype cast."""
    f1 = BinaryF1Score().to(device)
    total_t, n_imgs, n_batches = 0.0, 0, 0

    # Warmup over each distinct batch shape we'll see.
    seen = set()
    with torch.no_grad():
        for images, _ in loader:
            sh = tuple(images.shape)
            if sh in seen:
                continue
            seen.add(sh)
            x = prep(images.to("cuda", non_blocking=True))
            for _ in range(3):
                _ = fn(x)
            torch.cuda.synchronize()

    # Timed pass.
    with torch.no_grad():
        for images, masks in loader:
            x = prep(images.to("cuda", non_blocking=True))
            masks = masks.to("cuda", non_blocking=True)
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            logits = fn(x)
            torch.cuda.synchronize()
            total_t += time.perf_counter() - t0

            preds = torch.argmax(logits, dim=1)
            f1.update((preds > 0).long().flatten(), (masks > 0).long().flatten())
            n_imgs += images.size(0)
            n_batches += 1

    return {
        "label": label,
        "ms_batch": (total_t / max(n_batches, 1)) * 1000.0,
        "ms_image": (total_t / max(n_imgs, 1)) * 1000.0,
        "img_s": n_imgs / total_t,
        "dice": float(f1.compute().item()),
    }


# ----------------------------------------------------------------------------
# Sweep batch sizes. Same val_dataset, fresh DataLoader for each batch.
# ----------------------------------------------------------------------------
results_by_batch = {}
for bs in (2, 4, 8):
    print(f"\n--- loader batch={bs} ---")
    loader = DataLoader(
        val_dataset, batch_size=bs, shuffle=False,
        num_workers=2, pin_memory=(device.type == "cuda"),
        persistent_workers=True,
    )
    r_torch = time_path(loader, lambda x: torch_fp16_model(x),    lambda x: x.half(),  "torch-fp16")
    r_tfp16 = time_path(loader, lambda x: _chunked(trt_fp16_call, x), lambda x: x.float(), "trt-fp16")
    r_tint8 = time_path(loader, lambda x: _chunked(trt_int8_call, x), lambda x: x.float(), "trt-int8")
    results_by_batch[bs] = (r_torch, r_tfp16, r_tint8)

    for r in (r_torch, r_tfp16, r_tint8):
        print(f"  {r['label']:<11} {r['ms_batch']:>8.2f} ms/batch  "
              f"{r['ms_image']:>7.3f} ms/img  {r['img_s']:>6.1f} img/s  Dice={r['dice']:.4f}")

    del loader
    gc.collect()


# ----------------------------------------------------------------------------
# Decomposition: each step's contribution to the total speedup, per batch.
# ----------------------------------------------------------------------------
print("\n=== Decomposition: torch-fp16 -> trt-fp16 -> trt-int8 (per-image ms) ===")
print(f"  {'batch':>5} {'torch-fp16':>11} {'trt-fp16':>10} {'trt-int8':>10} | "
      f"{'fusion x':>9} {'IMMA x':>8} {'total x':>8}")
for bs, (rt, rf, ri) in results_by_batch.items():
    fusion = rt["ms_image"] / rf["ms_image"]
    imma   = rf["ms_image"] / ri["ms_image"]
    total  = rt["ms_image"] / ri["ms_image"]
    print(f"  {bs:>5} {rt['ms_image']:>11.3f} {rf['ms_image']:>10.3f} {ri['ms_image']:>10.3f} | "
          f"{fusion:>8.2f}x {imma:>7.2f}x {total:>7.2f}x")

print("\nDice (sanity, all three should match within ~1e-4):")
for bs, (rt, rf, ri) in results_by_batch.items():
    print(f"  batch={bs}: torch-fp16={rt['dice']:.4f}  trt-fp16={rf['dice']:.4f}  trt-int8={ri['dice']:.4f}")
